In [1]:
import os
import time

from SI.Practical_TM_SI import practical_TM_SI_randj
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Ananconda\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Ananconda\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Ananconda\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Ananconda\Lib\site-packages\tornado\platform\asyncio.py", line 205, in 

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [2]:
p, nt, ns, K = 100, 30, 50, 2
V = 3
sigma = 1.0
n_rep = 1000      
n_jobs = os.cpu_count()   # use every logical core

In [3]:
print(n_jobs)

20


In [4]:
rng = np.random.default_rng(2026)

X0 = rng.standard_normal((nt, p))
X_list = [rng.standard_normal((ns, p)) for _ in range(K)]

Sigma_0 = sigma ** 2 * np.eye(nt)
Sigma_K = [sigma ** 2 * np.eye(ns) for _ in range(K)]

In [5]:
def run(n_rep):
    p_values, branches, n_empty = [], [], 0
    start = time.perf_counter()

    for i in range(1, n_rep + 1):
        y0 = sigma * rng.standard_normal(nt)
        y_list = [sigma * rng.standard_normal(ns) for _ in range(K)]

        result = practical_TM_SI_randj(
            X0, y0, X_list, y_list, Sigma_0, Sigma_K,
            V=V, rng=rng, n_jobs=n_jobs,
        )

        if result['results']:
            p_values.append(result['results'][0]['p_value'])
            branches.append(result['branch'])
        else:
            n_empty += 1

        if i % 10 == 0:
            print(f"{i}/{n_rep}, "
                  f"p-values: {len(p_values)}, "
                  f"empty models: {n_empty}, "
                  f"time: {time.perf_counter() - start:.0f}s")

    return np.array(p_values), branches, n_empty

In [ ]:
list_p_value, branches, n_empty = run(n_rep)

10/1000, p-values: 2, empty models: 8, time: 26s
20/1000, p-values: 5, empty models: 15, time: 61s
30/1000, p-values: 7, empty models: 23, time: 81s
40/1000, p-values: 12, empty models: 28, time: 602s
50/1000, p-values: 16, empty models: 34, time: 654s
60/1000, p-values: 19, empty models: 41, time: 684s
70/1000, p-values: 23, empty models: 47, time: 734s


c:\Ananconda\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.412631e-08, tolerance: 2.605e-13
  model = cd_fast.enet_coordinate_descent(


80/1000, p-values: 28, empty models: 52, time: 3837s
90/1000, p-values: 30, empty models: 60, time: 3855s
100/1000, p-values: 35, empty models: 65, time: 3908s
110/1000, p-values: 38, empty models: 72, time: 3946s
120/1000, p-values: 45, empty models: 75, time: 4337s
130/1000, p-values: 48, empty models: 82, time: 4369s
140/1000, p-values: 52, empty models: 88, time: 4418s
150/1000, p-values: 57, empty models: 93, time: 4479s
160/1000, p-values: 62, empty models: 98, time: 4548s
170/1000, p-values: 66, empty models: 104, time: 4598s
180/1000, p-values: 71, empty models: 109, time: 4664s
190/1000, p-values: 77, empty models: 113, time: 8960s
200/1000, p-values: 82, empty models: 118, time: 9022s


In [ ]:
print(f"tests: {list_p_value.size}, empty models: {n_empty}")
print({b: branches.count(b) for b in sorted(set(branches))})

In [ ]:
alpha = 0.05
FPR = np.mean(list_p_value <= alpha)
print(f"FPR = {FPR}")

In [ ]:
statistic, p_value = stats.kstest(list_p_value, 'uniform')
print(f'KS-Test statistic = {statistic}')
print(f'KS-Test p-value = {p_value}')

In [ ]:
plt.hist(list_p_value, bins=20)

In [ ]:
plt.rcParams.update({'font.size': 18})
grid = np.linspace(0, 1, 101)
ecdf = np.searchsorted(np.sort(list_p_value), grid, side='right') / len(list_p_value)
plt.plot(grid, ecdf, 'r-', linewidth=6, label='p_value')
plt.plot([0, 1], [0, 1], 'k--')
plt.legend()
plt.tight_layout()
plt.show()